## Load Libraries

In [2]:
import pandas as pd
import re
import numpy as np


## Importing the data

In [25]:
plans = pd.read_csv("ks_wichita_2023_01_26.csv", low_memory=False)
df = pd.DataFrame(plans)

dataDict = pd.read_excel("Data Dictionary.xlsx")

## Initial analysis and summary

In [6]:
df.head()

,raw_row_number,date,time,location,lat,lng,geocode_source,subject_age,subject_race,subject_sex,...,violation,citation_issued,outcome,posted_speed,vehicle_color,vehicle_make,vehicle_model,vehicle_year,raw_defendant_race,raw_defendant_ethnicity
0,24860,2006-01-01,18:00:00,"1400 N MINNEAPOLIS, WICHITA, KS",37.708126,-97.315285,GM,NaN,unknown,NaN,...,ALLEY PARKING,True,citation,NaN,BURGUNDY OR MAROON,PONTIAC,NaN,NaN,U,NaN
1,28588,2006-01-01,18:00:00,"1410 S GLENDALE AVE, WICHITA, KS, 67218",37.666980,-97.277571,GM,NaN,unknown,NaN,...,PARK IN FIRE LANE,True,citation,NaN,GRAY,DODGE,INTREPID,NaN,U,NaN
2,25907,2006-01-01,18:00:00,"5300 S MOSLEY ST, WICHITA, KS, 67216",37.599499,-97.326801,GM,NaN,unknown,NaN,...,PARK W/IN 8' OF PRIVATE DRIVE,True,citation,NaN,BROWN,OLDSMOBILE,NaN,NaN,U,NaN
3,80832,2006-01-01,18:00:00,"E CENTRAL AVE, WICHITA, KS, 67214",37.693630,-97.315165,GM,24.0,white,male,...,NO DRIVER'S LICENSE; YIELD ROW-LEFT TURN,True,citation,NaN,WHITE,MERCURY,NaN,NaN,W,N
4,80815,2006-01-01,18:00:00,NaN,NaN,NaN,NaN,25.0,black,female,...,NO DRIVER'S LICENSE,True,citation,NaN,GRAY,CHEVROLET,NaN,NaN,B,N


In [7]:
preview = []

cols = df.columns

for col in cols:
    idx = dataDict.index[dataDict['Column name'].str.lower().str.strip() == col.lower().strip()]
    try:
        preview.append([col, df[col].dtypes, df[col].head(1).iloc[0],df[col].unique().size,
                        df[col].unique(), dataDict.loc[idx[0], 'Column meaning']])
    except IndexError:
        print(f"{col} not in df")
previewDf = pd.DataFrame(preview, columns = ['Variable', 'dtype', 'example', 'number of uniques', 'uniques', 'definition'])
previewDf

raw_defendant_race not in df
raw_defendant_ethnicity not in df


,Variable,dtype,example,number of uniques,uniques,definition
0,raw_row_number,object,24860,1030376,"[24860, 28588, 25907, 80832, 80815, 24485, 274...",An number used to join clean data back to the ...
1,date,object,2006-01-01,4884,"[2006-01-01, 2006-01-02, 2006-01-03, 2006-01-0...","The date of the stop, in YYYY-MM-DD format. So..."
2,time,object,18:00:00,1440,"[18:00:00, 18:01:00, 18:27:00, 18:45:00, 18:50...","The 24-hour time of the stop, in HH:MM format."
3,location,object,"1400 N MINNEAPOLIS, WICHITA, KS",47801,"[1400 N MINNEAPOLIS, WICHITA, KS, 1410 S GLEND...",The freeform text of the location. Occasionall...
4,lat,float64,37.708126,34064,"[37.7081257, 37.6669801, 37.599499, 37.6936296...",The latitude of the stop. If not provided by t...
5,lng,float64,-97.315285,31047,"[-97.3152848, -97.277571, -97.326801, -97.3151...",The longitude of the stop. If not provided by ...
6,geocode_source,object,GM,3,"[GM, nan, SU]",The geocoding service used to geocode the addr...
7,subject_age,float64,NaN,101,"[nan, 24.0, 25.0, 42.0, 33.0, 17.0, 18.0, 16.0...",The age of the stopped subject. When date of b...
8,subject_race,object,unknown,7,"[unknown, white, black, nan, asian/pacific isl...",The race of the stopped subject. Values are st...
9,subject_sex,object,NaN,3,"[nan, male, female]",The recorded sex of the stopped subject.


## Dataset cleaning

In [26]:
df = df[df['type'] == 'vehicular'] # want only vehicular citations
df = df.drop(columns = ['raw_row_number','lat', 'lng', 'geocode_source', 'citation_issued', 'outcome', 'vehicle_year', 'posted_speed']) # drop unwanted cols
df['year'] = pd.to_datetime(df['date']).dt.year # change dates to date/time type
df = df[df['year'] >= 2018] # filter years for greater than 2018

In [27]:
# Combine date + formatted time
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])

# Extract useful components
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month


In [28]:
# make location into zip
df['zip'] = df['location'].str.split(', ').str[-1]

In [29]:
# notes:
# raw_defendant_ethnicity doesn't provide any new/useful information and as of now is 100% empty so dropping
# subject_race and raw_defendant_race are the same but subject_race is initial (W) rather than the whole word (White)
# will use subject_race and drop raw_defendant_race

df = df.drop(columns = ['raw_defendant_ethnicity', 'raw_defendant_race'])

In [30]:
# cleaning the vehicle make column 

# noted that when a value had | in it with multiple makes (multi‑value field), the first make was chosen 

suffixes = ['CORP.', 'CO.', 'INC.', 'LTD.', 'MFG.', 'INTL.']

legacies = ["PONTIAC", "OLDSMOBILE", "PLYMOUTH", "SATURN", "DATSUN", "GEO", "HUMMER", "ROVER",
            "AUSTIN", "MG", "DAEWOO", "SUZULIGHT", "AVIA", "MURENA", "PEACE", "BRONCCO", "SABRA", "SCION", "SAAB", "VICTORIA",
           "AURORA", "MISTRAL", "PETERSON", "PANTHER"]

motorcycles = ["HARLEY-DAVIDSON", "DUCATI", "YAMAHA MOTOR CO.,", "KAWASAKI", "MOTO GUZZI", "INDIAN MOTORCYCLE", "VICTORY MOTORCYCLES",
               "INDIAN", "BUELL", "KTM", "TRIUMPH", "CAN-AM", "BASHAN", "HUFFY", "MINISCOOTER", "TAOTAO", "CHAPPY", "EAGLE",
              "STERLING INDUSTRIAL", "AURANTHETIC CHARGER", "MIDAS INTL", "RAMLIN", "SILVER BULLET", "BAJA" ]

# includes heavy trucks, trailers, industrial vehicles, buses, coaches and RVs
trailers = ["INTERNATIONAL COACH","UTILITY TRAILER", "LOAD RITE TRAILERS","INTERNATIONAL TRAILER", "SUN BLAZER TRAVEL TRAILER", "U-HAUL",
            "PELICAN ALUMINUM", "FRANCIS TRAVEL TRAILER","ALLEGRO MOTOR HOME", "TOYOCAR VAN CONTAINER TRAILER", "BUSHCRAFT TRAILER", 
            "FORTE TRAILER", "TRAILER COACH METAL SPEC.", "ELLIOTT MOBILE HOME", "LINCOLN ELECTRIC", "PETER PIRSCH & SONS", "FLXIBLE", "GILL MFG",
            "HEIL", "INTERCONSULT MFG", "GRONEWEGEN B.V.", "OTTERBACHER MFG", "SUBURBAN MOTORS", "FREIGHTLINER", 
            "PETERBILT", "MACK TRUCKS", "WESTERN STAR", "OSHKOSH MOTOR TRUCK", "UTILIMASTER",
            "GRUMMAN‑OLSEN", "SCHNURE HO' TRAILER", "P. J. TRAILER MFG", "EMPIRE TRAILER", "DOOLITTLE", "OTTERBACHER MFG", 
            "INTERCONSULT MFG", "GRONEWEGEN B.V.", "BLUEBIRD", "BLUE RIBBON COACH", "MOTOR COACH INDS.","COACH CRAFT", "JAYCO", "ALLEGRO MOTOR HOME",
            "HINO", "KENWORTH NORTHWEST", "KENWORTH MOTOR TRUCK", "INTERNATIONAL HARVESTER", "LANDCRAFT", "INTERNATIONAL INDS", "GRUMANN-OLSEN"]


def cleaned(row):
    row = str(row).strip()

    # if an unk/na, normalize all to the same category
    if pd.isna(row) or row.lower() in ["nan", "na", "unknown", "unk", "uk", "NA"]: # normalized all missing/unknown to UNKNOWN
        return 'UNKNOWN'

    # remaining are values that need cleaning
    row = row.split("(")[0].strip() # eg. JEEP (1989 TO PRESENT) becomes JEEP
    row = row.split("|")[0].strip()# eg. DODGE|HONDA becomes DODGE 
    
    if any(suffix in row for suffix in suffixes):
        row = " ".join(row.split(" ")[:-1]).strip()
        
    if any(legacy in row for legacy in legacies):
        row = 'LEGACY'

    if any(motorcycle in row for motorcycle in motorcycles):
        row = 'MOTORCYCLE'

    if any(trailer in row for trailer in trailers):
        row = 'TRAILER'

    replace = {
        "MERCEDES-BENZ": "MERCEDES",
        "MINI COOPER": "MINI",
        "LINCOLN-CONTINENTAL": "LINCOLN",
        "RANGE ROVER OF NORTH AMERICA": "LAND ROVER",
        "SMART CAR": "SMART",
        "SMARTLEE": "SMART",
        "PIAGGO": "PIAGGIO",
        "SUZULIGHT SU": "SUZULIGHT",
        "GENERAL MOTORS": "GM",
        "KIA MOTORS": "KIA",
        "PETERBILT MOTORS": "PETERBILT",
        "BUELL MOTOR": "BUELL",
        "MUSTANG": "FORD",
        "COOPER": "MINI",
        "PASSPORT":"HONDA",
        "CHARGER":"DODGE",
        "AVENGER":"DODGE",
        "TRANSIT":"FORD",
        "NA": "UNKNOWN",
        "N/A": "UNKNOWN",

    }
    
    return replace.get(row, row)

df['vehicle_make_clean'] = df['vehicle_make'].apply(cleaned)

In [31]:
# keeping dictionary at make level for clarity and simplicity, model would be used for cost/risk prediction, insurance, or fine‑grained segmentation
# color would be for behavior or aesthetic corrections

tier_map = {
    # Economy brands
    "CHEVROLET": "ECONOMY",
    "FORD": "ECONOMY",
    "TOYOTA": "ECONOMY",
    "HONDA": "ECONOMY",
    "DODGE": "ECONOMY",
    "NISSAN": "ECONOMY",
    "HYUNDAI": "ECONOMY",
    "KIA": "ECONOMY",
    "CHRYSLER": "ECONOMY",
    "BUICK": "ECONOMY",
    "MAZDA": "ECONOMY",
    "SUBARU": "ECONOMY",
    "MITSUBISHI": "ECONOMY",
    "SUZUKI": "ECONOMY",
    "ISUZU": "ECONOMY",
    "GM": "ECONOMY",   # collapsed General Motors

    # Mid-tier brands
    "VOLKSWAGEN": "MID",
    "MERCURY": "MID",
    "ACURA": "MID",
    "LEXUS": "MID",
    "INFINITI": "MID",
    "VOLVO": "MID",
    "AUDI": "MID",
    "MINI": "MID",
    "SMART": "MID",

    # Luxury brands
    "BMW": "LUXURY",
    "MERCEDES": "LUXURY",
    "CADILLAC": "LUXURY",
    "LINCOLN": "LUXURY",
    "JAGUAR": "LUXURY",
    "PORSCHE": "LUXURY",
    "FIAT": "LUXURY",        # borderline, but often treated as boutique
    "MASERATI": "LUXURY",
    "ALFA ROMEO": "LUXURY",
    "ROLLS-ROYCE": "LUXURY",
    "LOTUS": "LUXURY",
    "LAMBORGHINI": "LUXURY",
    "PIAGGIO": "LUXURY",     # scooters, but boutique
    "TRANSIT": "FORD",       # normalized earlier
    "CHARGER": "DODGE",      # normalized earlier

    # Buckets
    "LEGACY": "LEGACY",        # defunct car brands
    "MOTORCYCLE": "MOTORCYCLE",
    "TRAILER": "TRAILER",
    "UNKNOWN": "UNKNOWN"
}
df['vehicle_tier'] = df['vehicle_make_clean'].map(tier_map).fillna("UNKNOWN")

In [32]:
df = df.drop(columns = ["date", "time","location","type","vehicle_color","vehicle_make", "vehicle_model","datetime", "vehicle_make_clean"])

In [33]:
# get all unique violations to bin them

df['violation_norm'] = df['violation'].astype(str).apply(lambda x: re.sub(r"[|;/]", "|", x)) # normalize the deliminators to |
df['violation_list'] = df['violation_norm'].str.split("|")
df['violation_list'] = df['violation_list'].apply(lambda lst: [v.strip().upper() for v in lst if v.strip()])
all_violations = set(v for lst in df['violation_list'] for v in lst)
print(len(all_violations))


434


In [34]:
# create violation buckets (violation severities)
# when cleaning, if multipke violations will choose the first
# multiple are listed with | or ; or / or -

# severity ranking will be:
# High - immediate danger to life/legal consequences
# Medium severity - risky behavior but less catastrophic
# Low severity - administrative, parking, or minor equipment issues.

def score_violation(v):
    
    # High severity
    if re.search(r"DUI|DRUG|ALCOHOL|MARIJUANA|CONTROLLED SUBSTANCE|HIT AND RUN|HIT & RUN|ASSAULT|BATTERY|WEAPON|FIREARM|ELUDE|EVADE|REVOKED|NO DRIVER'S LICENSE|UNLICENSED|UNINSURED|FALSE INFORMATION|FRAUD|CRIMINAL", v):
        return 3
    
    # Medium severity
    if re.search(r"SPEED|SCHOOL ZONE|TEXTING|INATTENTIVE|RECKLESS|CARELESS|FOLLOW TOO CLOSE|UNSAFE|FAIL TO YIELD|FAIL TO REDUCE|STOP-SCHOOL BUS|GO AROUND RR GATE|DRIVE WRONG WAY|DRIVE OVER FIRE HOSE|SECURE LOAD|PROJECTING LOAD", v):
        return 2
    
    # Low severity
    if re.search(r"PARK|METER|TAG|PERMIT|PLACARD|SIGNAL|LIGHT|LAMP|WINDSHIELD|MIRROR|MUFFLER|EQUIPMENT|CURB|HYDRANT|SIDEWALK|PRIVATE LOT|NO TURN|NO U TURN|PLAYGROUND|SOUND|AMPLIFICATION|SOLICIT|BIKE|SCOOTER", v):
        return 1

    return 1 # default if all else fails
    
def worst_violation(vio_string):
    if pd.isna(vio_string):
        return 0

    cleaned = re.sub(r'[|;/]', '|', str(vio_string))
    
    violations = [v.strip() for v in cleaned.split("|") if v.strip()]
    
    # Assign severity, default to 1 if not found
    scored = [(v, score_violation(v)) for v in violations]

    # Pick violation with max severity
    worst = max(scored, key=lambda x: x[1])
    return worst[1]

df.loc[df['violation'].index, 'violation_severity'] = df['violation'].apply(worst_violation)

In [35]:
# clean dispositions column
df['disposition_norm'] = df['disposition'].astype(str).apply(lambda x: re.sub(r"[|;/]", "|", x)) # normalize the deliminators to |
df['disposition_list'] = df['disposition_norm'].str.split("|")
df['disposition_list'] = df['disposition_list'].apply(lambda lst: [v.strip().upper() for v in lst if v.strip()])
all_violations = set(v for lst in df['disposition_list'] for v in lst)
print(len(all_violations))
print(all_violations)

18
{'NOLLE PROS', 'HAD DRIVERS LICENSE', 'PRODUCED LICENSE', 'DEFERRED ACCEPTED', 'DISMISSED WITHOUT PREJUDICE', 'DISMISSED WITH PREJUDICE', 'DIVERSION', 'NAN', 'NOT GUILTY', 'GUILTY', 'DISMISSED', 'GUILTY (IVR)', 'PROVIDED PROOF', 'AMENDED CHARGE', 'GUILTY, FINES ABATED', 'REPAIRED', 'NA', 'NOLLO CONTENDRE'}


In [36]:
df = df.dropna(subset=['disposition'])

In [37]:
# create disposition binning (disposition_Severity)

"""

two bins align with Kansas law — conviction vs non‑conviction.
There isn’t a single Kansas statute that defines “disposition severity” for outcomes like dismissed, guilty, diversion, nolle pros, etc. 
Instead, these are standard court disposition codes used across U.S. courts to record the final status of a case. In Kansas, dispositions 
are governed by the Kansas Code of Criminal Procedure (K.S.A. Chapter 22) and related statutes (e.g., diversion agreements under K.S.A. 22‑2909,
dismissals under K.S.A. 60‑241). These codes don’t have a numeric severity scale in law; severity is usually inferred by analysts based on 
whether the disposition results in conviction, dismissal, diversion, or amendment


References for Disposition Codes
- Dismissals → Covered under K.S.A. 60‑241 (civil procedure, voluntary and involuntary dismissals).
- Diversion agreements → Defined in K.S.A. 22‑2909, allowing certain cases to be diverted from prosecution if conditions are met.
- Nolle prosequi (NOLLE PROS) → A prosecutor’s formal decision to drop charges; recognized in Kansas criminal procedure.
- Nolo contendere (NOLLO CONTENDRE) → A plea where the defendant does not contest the charge; treated like a guilty plea for sentencing.
- Deferred acceptance / diversion → Case is paused or diverted; if conditions are met, charges may be dismissed.
- Guilty / Not Guilty → Standard criminal dispositions under Kansas criminal procedure.
- Amended charge / fines abated / produced license / repaired → Administrative or corrective dispositions, often tied to municipal or traffic courts.

Disposition severity was collapsed to binary (conviction vs non‑conviction) to avoid quasi‑separation in MNLogit. 
Future work may revisit multinomial models if diversion cases are more frequent.”

"""

disposition_severity = {

    # high severity - conviction or an admission of guilt
    
    "GUILTY": 1, # full convistion, max consequence
    "GUILTY (IVR)": 1, # guilty via in‑vehicle recording or similar process, still a conviction.
    "GUILTY, FINES ABATED": 1, # conviction, but penalty reduced
    "NOLLO CONTENDRE": 1, # (no contest) → treated as guilty for sentencing, though not an admission.

    # low severity and medium severity - 
    # low severity - dismissal or compliance shown
    # medium severity - - conditional resolution or diversion
    
    "NOT GUILTY": 0, #acquittal, no conviction
    "DISMISSED": 0, # case dropped
    "DISMISSED WITH PREJUDICE": 0, # permanently dismissed, can't be refiled
    "DISMISSED WITHOUT PREJUDICE": 0, # dismissed but can be refiled
    "NOLLE PROS": 0, # (nolle prosequi) → prosecutor declines to pursue; case dropped
    "PROVIDED PROOF": 0, # defendant showed compliance (e.g., insurance, license)
    "PRODUCED LICENSE": 0, # compliance shown
    "HAD DRIVERS LICENSE": 0, # compliance shown
    "REPAIRED": 0, # equipment violation fixed
    "DEFERRED ACCEPTED": 0, # deferred judgment, accepted by court; conviction avoided if conditions met
    "DIVERSION": 0, # alternative program instead of conviction; moderate consequence
    "AMENDED CHARGE": 0, # charge reduced or changed; consequence depends on new charge

}

def worst_disposition(vio_string):
    if pd.isna(vio_string):
        return 0
    cleaned = re.sub(r'[|;/]', '|', vio_string)
    violations = [v.strip() for v in vio_string.split("|")if v.strip()]
    
    # Assign severity, default to 1 if not found
    scored = [(v, disposition_severity.get(v,1)) for v in violations]

    # Pick violation with max severity
    worst = max(scored, key=lambda x: x[1])
    return worst[1]


df.loc[df['disposition'].index, 'disposition_severity'] = df['disposition'].apply(worst_disposition)


In [38]:
# zip bucketing into city locations

"""
Zip breakdown:
https://usmapguide.com/kansas/wichita-zip-code-map/
https://zipcodes-us.com/zip/ks/wichita

- Central Wichita ZIPs (67202, 67203, 67214) cover downtown and historic districts.
- East Wichita ZIPs (67206, 67207, 67208, 67226, 67228, 67230) cover suburban growth and retail corridors.
- West Wichita ZIPs (67212, 67213, 67235) cover residential and newer developments.
- South Wichita ZIPs (67210, 67211, 67216, 67217) cover industrial and older residential areas.
- North Wichita ZIPs (67204, 67219, 67220, 67223) cover established neighborhoods and suburban expansion.

"""
central = ['67202', '67203', '67214']
east = ['67206', '67207', '67208', '67226', '67228', '67230']
west = ['67212', '67213', '67235']
south = ['67210', '67211', '67216', '67217']
north = ['67204', '67219', '67220', '67223']

df['city_location'] = np.where(df['zip'].isin(central), 'central', 
                               np.where(df['zip'].isin(east), 'east',
                               np.where(df['zip'].isin(west), 'west',
                               np.where(df['zip'].isin(south), 'south',
                               np.where(df['zip'].isin(north), 'north', 'unknown')))))

In [39]:
df['subject_race'] = df['subject_race'].fillna('unknown')

In [40]:
# Age categories with Unknown
bins = [0, 24, 34, 54, 64, 120]
labels = ['16-24', '25-34', '35-54', '55-64', '65+']
df['age_category'] = pd.cut(df['subject_age'], bins=bins, labels=labels, right=True)
df['age_category'] = df['age_category'].cat.add_categories('Unknown').fillna('Unknown')

# Sex and race with Unknown
df['subject_sex'] = df['subject_sex'].astype('category').cat.add_categories('Unknown').fillna('Unknown')

In [42]:
df = df.drop(columns = ['violation','subject_age','disposition','zip', 'violation_norm', 'violation_list', 'disposition_list','disposition_norm'])

In [43]:
# double checking there are no na vals

nullDf = []
null_data = df.isnull().sum()

# calculate percentages
percentages = (null_data / len(df)) * 100

# Create summary DataFrame
nullDf = pd.DataFrame({'Column': null_data.index,'Count': null_data.values,'Percentage': percentages.values})

print(nullDf.sort_values('Count', ascending=False))

                  Column  Count  Percentage
0           subject_race      0         0.0
1            subject_sex      0         0.0
2                   year      0         0.0
3                   hour      0         0.0
4            day_of_week      0         0.0
5                  month      0         0.0
6           vehicle_tier      0         0.0
7     violation_severity      0         0.0
8   disposition_severity      0         0.0
9          city_location      0         0.0
10          age_category      0         0.0


In [ ]:
# save the data
df.to_csv('cleanedWichita.csv')